# 📚 Bookcase Digitization - FAIL-SAFE INFERENCE
**Hướng dẫn:** Chạy 2 Cell duy nhất dưới đây. Đảm bảo đã bật GPU (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
# 1. DỌN DẸP VÀ CLONE
import os, shutil
%cd /content/
if os.path.exists('bookcase-digitization'): shutil.rmtree('bookcase-digitization')
!git clone https://github.com/pie-12/bookcase-digitization.git
%cd bookcase-digitization

# 2. CÀI ĐẶT THƯ VIỆN & VÁ LỖI TƯƠNG THÍCH
print("🛠 Đang cấu hình hệ thống...")
!pip install "numpy<2" opencv-python-headless==4.8.0.74 --force-reinstall -q
!pip install craft-text-detector vietocr==0.3.5 --no-deps -q
!pip install albumentations==1.4.2 einops gdown prefetch-generator shapely scikit-image -q
!git clone https://github.com/ultralytics/yolov5 -q

# 3. VÁ MÃ NGUỒN THƯ VIỆN BỊ LỖI (VGG16 & PILLOW)
print("🩹 Đang vá lỗi mã nguồn thư viện...")
# Fix CRAFT VGG issue
vgg_path = "/usr/local/lib/python3.12/dist-packages/craft_text_detector/models/basenet/vgg16_bn.py"
if os.path.exists(vgg_path):
    with open(vgg_path, 'r') as f: content = f.read()
    with open(vgg_path, 'w') as f: f.write(content.replace("from torchvision.models.vgg import model_urls", "model_urls = {}"))

# Fix VietOCR Pillow issue
vocr_path = "/usr/local/lib/python3.12/dist-packages/vietocr/tool/translate.py"
if os.path.exists(vocr_path):
    with open(vocr_path, 'r') as f: content = f.read()
    with open(vocr_path, 'w') as f: f.write(content.replace("Image.ANTIALIAS", "Image.Resampling.LANCZOS"))

print("✅ Hệ thống đã sẵn sàng!")

In [ ]:
# 4. TẢI MODEL VÀ CHẠY
import os
if not os.path.exists('last.pt') and not os.path.exists('best.pt'):
    from google.colab import files
    print("HÃY CHỌN FILE best.pt TỪ MÁY BẠN:")
    uploaded = files.upload()

print("🚀 ĐANG TRÍCH XUẤT...")
!python run_inference.py

import pandas as pd
if os.path.exists('final_results.csv'):
    print("🎉 THÀNH CÔNG!")
    display(pd.read_csv('final_results.csv'))
    from google.colab import files
    files.download('final_results.csv')
else:
    print("❌ Lỗi: Không tạo được file kết quả.")